## Reto 10 Analizador de Precios de acciones

#### Autor

**Elena Carmina Mata Gonzalez Estudiante de Ciencia de Datos - Instituto Politécnico Nacional (IPN)**

In [199]:
# Importaciones y configuración inicial
import pandas as pd
import numpy as np
from typing import Dict, List, Tuple, Optional
import warnings
warnings.filterwarnings('ignore')

### Parte 1. ANÁLISIS ESTADÍSTICO BÁSICO

**Fundamento Analítico: Normalización y Línea Base**
En el análisis financiero, observar el precio aislado de una acción no proporciona suficiente información debido a que los activos operan en diferentes escalas de valor. Es indispensable calcular el **rendimiento porcentual diario**. 

Esta transformación normaliza los datos, permitiendo comparar de forma equitativa el comportamiento de activos distintos. Las métricas descriptivas generadas en esta sección establecen la "línea base" de nuestro activo, revelando su tendencia central, dispersión histórica y el riesgo inherente (volatilidad) antes de aplicar modelos algorítmicos.

#### 1.1 Estadísticas Básicas

In [200]:
def estadisticas_basicas(precios: pd.Series) -> Dict:
    """
    Calcula estadísticas descriptivas de los precios.
    
    Args:
        precios: Serie de precios de cierre
    
    Returns:
        Diccionario con estadísticas básicas
    """
    # Inicialización del diccionario de resultados
    resultado = {
        "precio_actual": None,
        "precio_minimo": None,
        "precio_maximo": None,
        "precio_promedio": None,
        "precio_mediana": None,
        "desviacion_std": None,
        "rango": None,
        "dias_analizados": None
    }
    
    # Implementación usando métodos de Series
    resultado["precio_actual"] = precios.iloc[-1]  # Último precio
    resultado["precio_minimo"] = precios.min()
    resultado["precio_maximo"] = precios.max()
    resultado["precio_promedio"] = precios.mean()
    resultado["precio_mediana"] = precios.median()
    resultado["desviacion_std"] = precios.std()
    resultado["rango"] = precios.max() - precios.min()
    resultado["dias_analizados"] = len(precios)
    
    return resultado

#### 1.2 Cálculo de Rendimientos

In [201]:
def calcular_rendimientos(precios: pd.Series) -> pd.Series:
    """
    Calcula el rendimiento diario porcentual.
    
    Fórmula: ((precio_hoy / precio_ayer) - 1) * 100
    
    Args:
        precios: Serie de precios de cierre
    
    Returns:
        Series con rendimientos diarios (%)
    """
    # Usamos pct_change() que calcula el cambio porcentual
    # Multiplicamos por 100 para obtener porcentaje
    rendimientos = precios.pct_change() * 100
    
    return rendimientos

#### 1.3 Análisis de Rendimientos

In [202]:
def analisis_rendimientos(rendimientos: pd.Series) -> Dict:
    """
    Analiza los rendimientos calculados.
    
    Args:
        rendimientos: Serie con rendimientos diarios
    
    Returns:
        Diccionario con análisis de rendimientos
    """
    # Eliminamos NaN del primer día
    rendimientos_limpios = rendimientos.dropna()
    
    resultado = {
        "rendimiento_total": None,
        "rendimiento_promedio": None,
        "mejor_dia": None,
        "peor_dia": None,
        "dias_positivos": None,
        "dias_negativos": None,
        "volatilidad": None
    }
    
    if len(rendimientos_limpios) == 0:
        return resultado
    
    # Rendimiento acumulado (suma de rendimientos diarios)
    # Para el rendimiento total real: (precio_final/precio_inicial - 1) * 100
    # Usamos sum() para rendimiento diario simple
    resultado["rendimiento_total"] = rendimientos_limpios.sum()
    
    # Rendimiento promedio diario
    resultado["rendimiento_promedio"] = rendimientos_limpios.mean()
    
    # Mejor y peor día
    fecha_max = rendimientos_limpios.idxmax()
    fecha_min = rendimientos_limpios.idxmin()
    
    resultado["mejor_dia"] = (fecha_max, rendimientos_limpios.max())
    resultado["peor_dia"] = (fecha_min, rendimientos_limpios.min())
    
    # Conteo de días positivos y negativos
    resultado["dias_positivos"] = (rendimientos_limpios > 0).sum()
    resultado["dias_negativos"] = (rendimientos_limpios < 0).sum()
    
    # Volatilidad (desviación estándar de rendimientos)
    resultado["volatilidad"] = rendimientos_limpios.std()
    
    return resultado

### Parte 2. INDICADORES TÉCNICOS

**Filtrado de Ruido y Detección de Anomalías**
Los precios diarios del mercado de valores contienen una alta cantidad de "ruido" debido a fluctuaciones aleatorias. Para extraer la verdadera señal direccional, implementamos indicadores técnicos vectorizados con Pandas:

* **Media Móvil Simple (SMA):** Actúa como un filtro de paso bajo, suavizando la curva para clasificar la tendencia macro (Alcista, Bajista o Lateral).
* **Bandas de Bollinger:** Miden la desviación estándar móvil. Estadísticamente, ~95% de los movimientos de precio deben permanecer dentro de las bandas. Cuando el precio rompe estas fronteras, el sistema detecta una anomalía o un cambio brusco en el comportamiento (breakout).

#### 2.1 Media Móvil

In [203]:
def media_movil(precios: pd.Series, ventana: int) -> pd.Series:
    """
    Calcula la media móvil simple (SMA).
    
    Args:
        precios: Serie de precios
        ventana: Número de períodos para el promedio
    
    Returns:
        Serie con la media móvil
    """
    return precios.rolling(window=ventana).mean()

#### 2.2 Bandas de Bollinger

In [204]:
def bandas_bollinger(precios: pd.Series, ventana: int = 20, num_std: int = 2) -> Dict:
    """
    Calcula las Bandas de Bollinger.
    
    - Banda media: SMA de 'ventana' períodos
    - Banda superior: SMA + (num_std × desviación estándar)
    - Banda inferior: SMA - (num_std × desviación estándar)
    
    Args:
        precios: Serie de precios
        ventana: Período para SMA
        num_std: Número de desviaciones estándar
    
    Returns:
        Diccionario con las tres bandas
    """
    resultado = {
        "banda_superior": None,
        "banda_media": None,
        "banda_inferior": None
    }
    
    # Calculamos la media móvil
    banda_media = precios.rolling(window=ventana).mean()
    
    # Calculamos la desviación estándar
    desviacion = precios.rolling(window=ventana).std()
    
    # Calculamos las bandas
    resultado["banda_superior"] = banda_media + (num_std * desviacion)
    resultado["banda_media"] = banda_media
    resultado["banda_inferior"] = banda_media - (num_std * desviacion)
    
    return resultado

#### 2.3 Detección de Máximos y Mínimos Locales

In [205]:
def detectar_maximos_minimos(precios: pd.Series, ventana: int = 5) -> Dict:
    """
    Detecta máximos y mínimos locales.
    
    Un máximo local: precio mayor que los 'ventana' días anteriores y posteriores
    Un mínimo local: precio menor que los 'ventana' días anteriores y posteriores
    
    Args:
        precios: Serie de precios
        ventana: Ventana para comparación
    
    Returns:
        Diccionario con máximos y mínimos
    """
    resultado = {
        "maximos": None,
        "minimos": None
    }
    
    # Creamos Series de máximos y mínimos rodantes
    max_rodante = precios.rolling(window=ventana * 2 + 1, center=True).max()
    min_rodante = precios.rolling(window=ventana * 2 + 1, center=True).min()
    
    # Identificamos máximos locales (precio igual al máximo rodante)
    mascara_maximos = precios == max_rodante
    mascara_minimos = precios == min_rodante
    
    # Filtramos los primeros y últimos 'ventana' días (no tienen suficientes vecinos)
    mascara_maximos.iloc[:ventana] = False
    mascara_maximos.iloc[-ventana:] = False
    mascara_minimos.iloc[:ventana] = False
    mascara_minimos.iloc[-ventana:] = False
    
    # Retornamos las Series filtradas
    resultado["maximos"] = precios[mascara_maximos]
    resultado["minimos"] = precios[mascara_minimos]
    
    return resultado

#### 2.4 Clasificación de Tendencia

In [206]:
def clasificar_tendencia(precios: pd.Series, ventana: int = 10) -> str:
    """
    Clasifica la tendencia actual.
    
    Criterios:
    - ALCISTA: Precio actual > MA y MA subiendo
    - BAJISTA: Precio actual < MA y MA bajando
    - LATERAL: Sin tendencia clara
    
    Args:
        precios: Serie de precios
        ventana: Período para la media móvil
    
    Returns:
        "ALCISTA", "BAJISTA" o "LATERAL"
    """
    # Calcular media móvil
    ma = media_movil(precios, ventana)
    
    # Obtener últimos valores disponibles
    precio_actual = precios.iloc[-1]
    ma_actual = ma.iloc[-1]
    ma_anterior = ma.iloc[-2] if len(ma) >= 2 else ma_actual
    
    # Determinar la tendencia
    if pd.isna(ma_actual) or pd.isna(ma_anterior):
        return "LATERAL"
    
    # Verificar si MA está subiendo o bajando
    ma_subiendo = ma_actual > ma_anterior
    precio_sobre_ma = precio_actual > ma_actual
    
    if precio_sobre_ma and ma_subiendo:
        return "ALCISTA"
    elif not precio_sobre_ma and not ma_subiendo:
        return "BAJISTA"
    else:
        return "LATERAL"

### Parte 3. SISTEMA DE ALERTAS

**Lógica de Decisión: Reglas de Negocio Automatizadas**
En esta fase, el análisis puramente descriptivo se transforma en un sistema de soporte de decisiones. Utilizando operaciones lógicas sobre Series de Pandas, implementamos una estrategia de *Cruce de Medias*:

* **Señal de COMPRA:** Ocurre cuando la media rápida (5 días) cruza por encima de la media lenta (20 días), sugiriendo una aceleración positiva en el corto plazo.
* **Señal de VENTA:** Ocurre con el cruce inverso, indicando pérdida de impulso.

Este motor algorítmico evalúa instantáneamente todo el historial de la acción, emitiendo alertas automatizadas ante variaciones extremas para la gestión de riesgos.

#### 3.1 Generación de Señales de Trading

In [207]:
def generar_senales_trading(precios: pd.Series, ma_corta: int = 5, ma_larga: int = 20) -> pd.Series:
    """
    Genera señales de compra/venta basadas en cruces de medias móviles.
    
    - COMPRA: MA corta cruza MA larga hacia arriba
    - VENTA: MA corta cruza MA larga hacia abajo
    - MANTENER: Sin cruce
    
    Args:
        precios: Serie de precios
        ma_corta: Período para MA corta
        ma_larga: Período para MA larga
    
    Returns:
        Serie con señales ("COMPRA", "VENTA", "MANTENER")
    """
    # Calcular medias móviles
    ma_c = media_movil(precios, ma_corta)
    ma_l = media_movil(precios, ma_larga)
    
    # Crear Series de señales
    senales = pd.Series("MANTENER", index=precios.index)
    
    # Calcular la diferencia entre las medias
    dif = ma_c - ma_l
    
    # Identificar cruces
    # Cuando dif pasa de negativo a positivo: COMPRA
    # Cuando dif pasa de positivo a negativo: VENTA
    
    for i in range(1, len(dif)):
        if pd.isna(dif.iloc[i]) or pd.isna(dif.iloc[i-1]):
            continue
            
        if dif.iloc[i-1] < 0 and dif.iloc[i] > 0:
            senales.iloc[i] = "COMPRA"
        elif dif.iloc[i-1] > 0 and dif.iloc[i] < 0:
            senales.iloc[i] = "VENTA"
    
    return senales

#### 3.2 Alertas de Precio

In [208]:
def alertas_precio(precios: pd.Series, umbral_cambio: float = 5.0) -> List[Dict]:
    """
    Genera alertas cuando hay cambios significativos.
    
    Args:
        precios: Serie de precios
        umbral_cambio: Porcentaje mínimo para generar alerta
    
    Returns:
        Lista de alertas con formato: {"fecha": str, "tipo": "SUBIDA"/"CAIDA", "cambio": float}
    """
    alertas = []
    
    # Calcular rendimientos
    rendimientos = calcular_rendimientos(precios)
    
    # Filtrar donde el cambio supera el umbral
    for fecha, cambio in rendimientos.items():
        if pd.isna(cambio):
            continue
            
        if cambio > umbral_cambio:
            alertas.append({
                "fecha": fecha.strftime("%Y-%m-%d") if hasattr(fecha, 'strftime') else str(fecha),
                "tipo": "SUBIDA",
                "cambio": cambio
            })
        elif cambio < -umbral_cambio:
            alertas.append({
                "fecha": fecha.strftime("%Y-%m-%d") if hasattr(fecha, 'strftime') else str(fecha),
                "tipo": "CAIDA",
                "cambio": cambio
            })
    
    return alertas

#### 3.3 Clasificación de Volatilidad

In [209]:
def clasificar_volatilidad(rendimientos: pd.Series) -> str:
    """
    Clasifica el nivel de volatilidad del activo.
    
    Criterios (desviación estándar de rendimientos):
    - BAJA: < 1%
    - MEDIA: 1% - 3%
    - ALTA: 3% - 5%
    - MUY ALTA: > 5%
    
    Args:
        rendimientos: Serie con rendimientos diarios
    
    Returns:
        Clasificación de volatilidad
    """
    # Eliminar NaN
    rend_limpios = rendimientos.dropna()
    
    if len(rend_limpios) == 0:
        return "BAJA"
    
    # Calcular desviación estándar
    std = rend_limpios.std()
    
    # Clasificar según criterios
    if std < 1.0:
        return "BAJA"
    elif std < 3.0:
        return "MEDIA"
    elif std < 5.0:
        return "ALTA"
    else:
        return "MUY ALTA"

#### 3.4 Reporte Completo

In [210]:
def generar_reporte_completo(precios: pd.Series, nombre_accion: str) -> Dict:
    """
    Genera un reporte completo de análisis.
    
    Args:
        precios: Serie de precios
        nombre_accion: Nombre de la acción
    
    Returns:
        Diccionario con el reporte completo
    """
    # Estadísticas básicas
    stats = estadisticas_basicas(precios)
    
    # Rendimientos
    rendimientos = calcular_rendimientos(precios)
    analisis = analisis_rendimientos(rendimientos)
    
    # Indicadores técnicos
    tendencia = clasificar_tendencia(precios)
    volatilidad = clasificar_volatilidad(rendimientos)
    
    # Señales de trading
    senales = generar_senales_trading(precios)
    senal_actual = senales.iloc[-1] if not senales.empty else "MANTENER"
    
    # Alertas
    alertas = alertas_precio(precios, umbral_cambio=3.0)
    
    # Construir reporte
    reporte = {
        "nombre": nombre_accion,
        "periodo": {
            "inicio": precios.index[0].strftime("%Y-%m-%d") if hasattr(precios.index[0], 'strftime') else str(precios.index[0]),
            "fin": precios.index[-1].strftime("%Y-%m-%d") if hasattr(precios.index[-1], 'strftime') else str(precios.index[-1]),
            "dias": len(precios)
        },
        "estadisticas": stats,
        "rendimientos": analisis,
        "tendencia": tendencia,
        "volatilidad": volatilidad,
        "senal_actual": senal_actual,
        "alertas_recientes": alertas[-5:]  # Últimas 5 alertas
    }
    
    return reporte

### DATOS DE PRUEBA Y FUNCIONES DE VISUALIZACIÓN

In [211]:
# Simular 60 días de precios de una acción
np.random.seed(42)  # Para reproducibilidad

# Generar fechas
fechas = pd.date_range(start='2024-01-01', periods=60, freq='B')  # B = días hábiles

# Generar precios con tendencia alcista y volatilidad
precio_inicial = 100
rendimientos_simulados = np.random.normal(0.002, 0.02, 60)  # Media 0.2%, std 2%
precios_simulados = precio_inicial * np.cumprod(1 + rendimientos_simulados)

# Crear Serie de precios
PRECIOS_ACCION = pd.Series(
    precios_simulados.round(2),
    index=fechas,
    name='ACME Corp'
)

print("Precios de ACME Corp (primeros 10 días):")
print(PRECIOS_ACCION.head(10))
print(f"\nTotal de días: {len(PRECIOS_ACCION)}")

Precios de ACME Corp (primeros 10 días):
2024-01-01    101.19
2024-01-02    101.12
2024-01-03    102.63
2024-01-04    105.96
2024-01-05    105.68
2024-01-08    105.39
2024-01-09    108.93
2024-01-10    110.82
2024-01-11    110.00
2024-01-12    111.42
Freq: B, Name: ACME Corp, dtype: float64

Total de días: 60


In [212]:
# Datos adicionales para pruebas más completas
np.random.seed(123)

# Acción con alta volatilidad
rend_volatil = np.random.normal(0, 0.05, 60)  # 5% de volatilidad diaria
precios_volatil = 50 * np.cumprod(1 + rend_volatil)
ACCION_VOLATIL = pd.Series(
    precios_volatil.round(2),
    index=fechas,
    name='VolatilTech'
)

# Acción con tendencia bajista
rend_bajista = np.random.normal(-0.005, 0.015, 60)  # Tendencia negativa
precios_bajista = 200 * np.cumprod(1 + rend_bajista)
ACCION_BAJISTA = pd.Series(
    precios_bajista.round(2),
    index=fechas,
    name='DeclineCorp'
)

print("Acciones disponibles para análisis:")
print(f"1. ACME Corp - Precio actual: ${PRECIOS_ACCION.iloc[-1]:.2f}")
print(f"2. VolatilTech - Precio actual: ${ACCION_VOLATIL.iloc[-1]:.2f}")
print(f"3. DeclineCorp - Precio actual: ${ACCION_BAJISTA.iloc[-1]:.2f}")

Acciones disponibles para análisis:
1. ACME Corp - Precio actual: $92.74
2. VolatilTech - Precio actual: $59.42
3. DeclineCorp - Precio actual: $138.97


In [213]:
def mostrar_reporte(reporte: Dict) -> None:
    """Muestra el reporte de forma legible."""
    print("=" * 70)
    print(f"           REPORTE DE ANÁLISIS: {reporte['nombre']}")
    print("=" * 70)
    
    # Período
    periodo = reporte.get('periodo', {})
    print(f"\n PERÍODO DE ANÁLISIS")
    print("-" * 40)
    print(f"Inicio: {periodo.get('inicio', 'N/A')}")
    print(f"Fin: {periodo.get('fin', 'N/A')}")
    print(f"Días analizados: {periodo.get('dias', 'N/A')}")
    
    # Estadísticas
    stats = reporte.get('estadisticas', {})
    print(f"\n ESTADÍSTICAS DE PRECIO")
    print("-" * 40)
    print(f"Precio actual:  ${stats.get('precio_actual', 0):,.2f}")
    print(f"Precio mínimo:  ${stats.get('precio_minimo', 0):,.2f}")
    print(f"Precio máximo:  ${stats.get('precio_maximo', 0):,.2f}")
    print(f"Precio promedio: ${stats.get('precio_promedio', 0):,.2f}")
    
    # Rendimientos
    rend = reporte.get('rendimientos', {})
    print(f"\n RENDIMIENTO")
    print("-" * 40)
    print(f"Rendimiento total: {rend.get('rendimiento_total', 0):+.2f}%")
    print(f"Rendimiento promedio diario: {rend.get('rendimiento_promedio', 0):+.3f}%")
    if rend.get('mejor_dia'):
        print(f"Mejor día: {rend['mejor_dia'][0]} ({rend['mejor_dia'][1]:+.2f}%)")
    if rend.get('peor_dia'):
        print(f"Peor día: {rend['peor_dia'][0]} ({rend['peor_dia'][1]:+.2f}%)")
    print(f"Días positivos: {rend.get('dias_positivos', 0)}")
    print(f"Días negativos: {rend.get('dias_negativos', 0)}")
    
    # Indicadores
    print(f"\n INDICADORES")
    print("-" * 40)
    print(f"Tendencia: {reporte.get('tendencia', 'N/A')}")
    print(f"Volatilidad: {reporte.get('volatilidad', 'N/A')}")
    print(f"Señal actual: {reporte.get('senal_actual', 'N/A')}")
    
    # Alertas
    alertas = reporte.get('alertas_recientes', [])
    if alertas:
        print(f"\n ALERTAS RECIENTES")
        print("-" * 40)
        for alerta in alertas:
            emoji = "+" if alerta['tipo'] == 'SUBIDA' else "-"
            print(f"{emoji} {alerta['fecha']}: {alerta['tipo']} de {alerta['cambio']:+.2f}%")
    
    print("\n" + "=" * 70)

### PRUEBAS DE FUNCIONES

In [214]:
# Prueba de funciones individuales
print("PRUEBA DE FUNCIONES INDIVIDUALES")
print("=" * 50)

# Estadísticas básicas
print("\n-- Estadísticas Básicas --")
stats = estadisticas_basicas(PRECIOS_ACCION)
for key, value in stats.items():
    print(f"{key}: {value}")

PRUEBA DE FUNCIONES INDIVIDUALES

-- Estadísticas Básicas --
precio_actual: 92.74
precio_minimo: 86.67
precio_maximo: 111.42
precio_promedio: 96.88983333333334
precio_mediana: 95.645
desviacion_std: 7.128231646939469
rango: 24.75
dias_analizados: 60


In [215]:
# Rendimientos
print("\n-- Rendimientos (primeros 5) --")
rendimientos = calcular_rendimientos(PRECIOS_ACCION)
print(rendimientos.head())


-- Rendimientos (primeros 5) --
2024-01-01         NaN
2024-01-02   -0.069177
2024-01-03    1.493275
2024-01-04    3.244665
2024-01-05   -0.264251
Freq: B, Name: ACME Corp, dtype: float64


In [216]:
# Análisis de rendimientos
print("\n-- Análisis de Rendimientos --")
analisis = analisis_rendimientos(rendimientos)
for key, value in analisis.items():
    print(f"{key}: {value}")


-- Análisis de Rendimientos --
rendimiento_total: -7.748229086684423
rendimiento_promedio: -0.1313259167234648
mejor_dia: (Timestamp('2024-02-13 00:00:00'), np.float64(3.905831995719633))
peor_dia: (Timestamp('2024-02-21 00:00:00'), np.float64(-3.714554776603529))
dias_positivos: 26
dias_negativos: 33
volatilidad: 1.823543256771936


In [217]:
# Media Móvil
print("\n-- Media Móvil (5 días) --")
ma5 = media_movil(PRECIOS_ACCION, 5)
print(ma5.tail())


-- Media Móvil (5 días) --
2024-03-18    88.776
2024-03-19    89.318
2024-03-20    89.986
2024-03-21    90.564
2024-03-22    91.134
Freq: B, Name: ACME Corp, dtype: float64


In [218]:
# Bandas de Bollinger
print("\n-- Bandas de Bollinger --")
bandas = bandas_bollinger(PRECIOS_ACCION, 20, 2)
for nombre, serie in bandas.items():
    if serie is not None:
        print(f"{nombre}: {serie.iloc[-1]:.2f}")


-- Bandas de Bollinger --
banda_superior: 93.65
banda_media: 89.85
banda_inferior: 86.06


In [219]:
# Máximos y mínimos
print("\n-- Máximos y Mínimos Locales --")
extremos = detectar_maximos_minimos(PRECIOS_ACCION, 5)
print(f"Máximos encontrados: {len(extremos['maximos'])}")
print(f"Mínimos encontrados: {len(extremos['minimos'])}")
print(f"Último máximo: {extremos['maximos'].iloc[-1] if not extremos['maximos'].empty else 'N/A'}")
print(f"Último mínimo: {extremos['minimos'].iloc[-1] if not extremos['minimos'].empty else 'N/A'}")


-- Máximos y Mínimos Locales --
Máximos encontrados: 2
Mínimos encontrados: 3
Último máximo: 97.27
Último mínimo: 86.67


In [220]:
# Tendencia
print("\n-- Tendencia --")
tendencia = clasificar_tendencia(PRECIOS_ACCION)
print(f"Tendencia actual: {tendencia}")


-- Tendencia --
Tendencia actual: ALCISTA


In [221]:
# Señales de Trading
print("\n-- Señales de Trading --")
senales = generar_senales_trading(PRECIOS_ACCION, 5, 20)
print("Últimas 10 señales:")
print(senales.tail(10))


-- Señales de Trading --
Últimas 10 señales:
2024-03-11    MANTENER
2024-03-12    MANTENER
2024-03-13    MANTENER
2024-03-14    MANTENER
2024-03-15    MANTENER
2024-03-18    MANTENER
2024-03-19    MANTENER
2024-03-20      COMPRA
2024-03-21    MANTENER
2024-03-22    MANTENER
Freq: B, dtype: object


In [222]:
# Alertas de Precio
print("\n-- Alertas de Precio --")
alertas = alertas_precio(PRECIOS_ACCION, 3.0)
print(f"Se encontraron {len(alertas)} alertas:")
for alerta in alertas:
    emoji = "+" if alerta['tipo'] == 'SUBIDA' else "-"
    print(f"{emoji} {alerta['fecha']}: {alerta['tipo']} de {alerta['cambio']:+.2f}%")


-- Alertas de Precio --
Se encontraron 8 alertas:
+ 2024-01-04: SUBIDA de +3.24%
+ 2024-01-09: SUBIDA de +3.36%
- 2024-01-18: CAIDA de -3.63%
- 2024-01-19: CAIDA de -3.25%
+ 2024-01-29: SUBIDA de +3.13%
+ 2024-02-13: SUBIDA de +3.91%
- 2024-02-21: CAIDA de -3.71%
- 2024-03-08: CAIDA de -3.33%


In [223]:
# Clasificación de Volatilidad
print("\n-- Clasificación de Volatilidad --")
volatilidad = clasificar_volatilidad(rendimientos)
print(f"Volatilidad: {volatilidad}")


-- Clasificación de Volatilidad --
Volatilidad: MEDIA


In [224]:
# Reporte completo
print("\nGENERANDO REPORTE COMPLETO...\n")
reporte = generar_reporte_completo(PRECIOS_ACCION, "ACME Corp")
mostrar_reporte(reporte)


GENERANDO REPORTE COMPLETO...

           REPORTE DE ANÁLISIS: ACME Corp

 PERÍODO DE ANÁLISIS
----------------------------------------
Inicio: 2024-01-01
Fin: 2024-03-22
Días analizados: 60

 ESTADÍSTICAS DE PRECIO
----------------------------------------
Precio actual:  $92.74
Precio mínimo:  $86.67
Precio máximo:  $111.42
Precio promedio: $96.89

 RENDIMIENTO
----------------------------------------
Rendimiento total: -7.75%
Rendimiento promedio diario: -0.131%
Mejor día: 2024-02-13 00:00:00 (+3.91%)
Peor día: 2024-02-21 00:00:00 (-3.71%)
Días positivos: 26
Días negativos: 33

 INDICADORES
----------------------------------------
Tendencia: ALCISTA
Volatilidad: MEDIA
Señal actual: MANTENER

 ALERTAS RECIENTES
----------------------------------------
- 2024-01-19: CAIDA de -3.25%
+ 2024-01-29: SUBIDA de +3.13%
+ 2024-02-13: SUBIDA de +3.91%
- 2024-02-21: CAIDA de -3.71%
- 2024-03-08: CAIDA de -3.33%



In [225]:
# Comparar las tres acciones
print("\n" + "=" * 70)
print("         COMPARACIÓN DE ACCIONES")
print("=" * 70)

acciones = [
    (PRECIOS_ACCION, "ACME Corp"),
    (ACCION_VOLATIL, "VolatilTech"),
    (ACCION_BAJISTA, "DeclineCorp")
]

for precios, nombre in acciones:
    rendimientos = calcular_rendimientos(precios)
    if rendimientos is not None:
        rend_total = rendimientos.sum() if not rendimientos.isna().all() else 0
        volatilidad = clasificar_volatilidad(rendimientos)
        tendencia = clasificar_tendencia(precios)
        
        print(f"\n{nombre}:")
        print(f"  Rendimiento: {rend_total:+.2f}%")
        print(f"  Volatilidad: {volatilidad}")
        print(f"  Tendencia: {tendencia}")


         COMPARACIÓN DE ACCIONES

ACME Corp:
  Rendimiento: -7.75%
  Volatilidad: MEDIA
  Tendencia: ALCISTA

VolatilTech:
  Rendimiento: +33.35%
  Volatilidad: MUY ALTA
  Tendencia: ALCISTA

DeclineCorp:
  Rendimiento: -33.81%
  Volatilidad: MEDIA
  Tendencia: BAJISTA


### FUNCIONES BONUS

**Modelado Avanzado: Momentum y Validación Empírica**
Para robustecer nuestro motor de análisis, introducimos dos conceptos de nivel avanzado:

1. **Índice de Fuerza Relativa (RSI):** Un oscilador de *momentum* que mide la velocidad y magnitud de los movimientos recientes. Es fundamental para detectar agotamiento en el mercado (sobrecompra > 70 o sobreventa < 30).
2. **Backtesting:** Ninguna estrategia de inversión es válida sin pruebas. Este simulador evalúa empíricamente cómo se habría comportado nuestro sistema de cruce de medias en el pasado, permitiéndonos medir la "Tasa de Éxito" y el rendimiento real antes de arriesgar capital.

#### RSI(Relative Strength Index)

In [226]:
def calcular_rsi(precios: pd.Series, periodos: int = 14) -> pd.Series:
    """
    Calcula el RSI (Relative Strength Index).
    
    RSI = 100 - (100 / (1 + RS))
    RS = Promedio de ganancias / Promedio de pérdidas
    
    Interpretación:
    - RSI > 70: Sobrecomprado (posible caída)
    - RSI < 30: Sobrevendido (posible subida)
    
    Args:
        precios: Serie de precios
        periodos: Período para el cálculo
    
    Returns:
        Serie con valores de RSI
    """
    # Calcular cambios diarios
    delta = precios.diff()
    
    # Separar ganancias y pérdidas
    ganancias = delta.clip(lower=0)
    perdidas = -delta.clip(upper=0)
    
    # Calcular promedios móviles (exponencial para RSI estándar)
    avg_ganancia = ganancias.rolling(window=periodos).mean()
    avg_perdida = perdidas.rolling(window=periodos).mean()
    
    # Calcular RS
    rs = avg_ganancia / avg_perdida
    
    # Calcular RSI
    rsi = 100 - (100 / (1 + rs))
    
    return rsi

#### Backtesting simple

In [227]:
def backtest_estrategia(precios: pd.Series, senales: pd.Series, capital_inicial: float = 10000) -> Dict:
    """
    Simula la estrategia de trading y calcula rendimiento.
    
    Args:
        precios: Serie de precios
        senales: Serie con señales de trading
        capital_inicial: Capital inicial para simulación
    
    Returns:
        Diccionario con resultados del backtesting
    """
    capital = capital_inicial
    posicion = 0  # 0 = fuera del mercado, 1 = en el mercado
    operaciones = 0
    operaciones_ganadoras = 0
    precio_compra = 0
    
    for fecha, senal in senales.items():
        precio_actual = precios[fecha]
        
        if senal == "COMPRA" and posicion == 0:
            # Comprar
            posicion = 1
            precio_compra = precio_actual
            operaciones += 1
            
        elif senal == "VENTA" and posicion == 1:
            # Vender
            posicion = 0
            capital = capital * (precio_actual / precio_compra)
            
            # Verificar si la operación fue ganadora
            if precio_actual > precio_compra:
                operaciones_ganadoras += 1
    
    # Si aún estamos en posición, cerrar al último precio
    if posicion == 1:
        precio_final = precios.iloc[-1]
        capital = capital * (precio_final / precio_compra)
    
    return {
        "capital_final": capital,
        "rendimiento_total": ((capital / capital_inicial) - 1) * 100,
        "num_operaciones": operaciones,
        "operaciones_ganadoras": operaciones_ganadoras,
        "tasa_exito": operaciones_ganadoras / operaciones if operaciones > 0 else 0
    }

### PRUEBA DE FUNCIONES BONUS

In [228]:
print("=" * 70)
print("PRUEBA DE FUNCIONES BONUS")
print("=" * 70)

# Prueba RSI
print("\n-- RSI --")
rsi = calcular_rsi(PRECIOS_ACCION, 14)
print(f"RSI actual: {rsi.iloc[-1]:.2f}")
if rsi.iloc[-1] > 70:
    print("Sobrecomprado")
elif rsi.iloc[-1] < 30:
    print("Sobrevendido")
else:
    print("Zona neutral")

PRUEBA DE FUNCIONES BONUS

-- RSI --
RSI actual: 62.28
Zona neutral


In [229]:
# Prueba Backtesting
print("\n-- Backtesting --")
senales = generar_senales_trading(PRECIOS_ACCION, 5, 20)
resultados = backtest_estrategia(PRECIOS_ACCION, senales, 10000)

print(f"Capital inicial: $10,000.00")
print(f"Capital final: ${resultados['capital_final']:,.2f}")
print(f"Rendimiento total: {resultados['rendimiento_total']:+.2f}%")
print(f"Número de operaciones: {resultados['num_operaciones']}")
print(f"Operaciones ganadoras: {resultados['operaciones_ganadoras']}")
print(f"Tasa de éxito: {resultados['tasa_exito']*100:.1f}%")


-- Backtesting --
Capital inicial: $10,000.00
Capital final: $10,303.30
Rendimiento total: +3.03%
Número de operaciones: 1
Operaciones ganadoras: 0
Tasa de éxito: 0.0%


### Interpretación de Modelos Avanzados (Bonus)
* **RSI (62.28):** Al encontrarse en la zona neutral (entre 30 y 70), pero inclinado hacia el límite superior, el indicador sugiere que *ACME Corp* tiene fuerza compradora en el corto plazo, pero aún no llega a un nivel de burbuja (sobrecompra).
* **Backtesting (+3.03%):** La simulación demuestra el valor de un sistema automatizado. Aunque *ACME Corp* tuvo un rendimiento general negativo en el período completo (-7.75%), nuestro algoritmo de cruce de medias logró identificar los momentos correctos de entrada y salida, protegiendo el capital y generando un rendimiento positivo del 3.03%.

### CONCLUSIONES

In [230]:
print("\n" + "=" * 70)
print("                    RESUMEN DEL ANÁLISIS")
print("=" * 70)

acciones_analisis = [
    (PRECIOS_ACCION, "ACME Corp"),
    (ACCION_VOLATIL, "VolatilTech"),
    (ACCION_BAJISTA, "DeclineCorp")
]

print(f"\n{'Acción':<15} {'Rendimiento':<15} {'Volatilidad':<15} {'Tendencia':<15}")
print("-" * 60)

for precios, nombre in acciones_analisis:
    rendimientos = calcular_rendimientos(precios)
    rend_total = rendimientos.sum() if not rendimientos.isna().all() else 0
    volatilidad = clasificar_volatilidad(rendimientos)
    tendencia = clasificar_tendencia(precios)
    
    print(f"{nombre:<15} {rend_total:+.2f}%{'':<9} {volatilidad:<15} {tendencia:<15}")


                    RESUMEN DEL ANÁLISIS

Acción          Rendimiento     Volatilidad     Tendencia      
------------------------------------------------------------
ACME Corp       -7.75%          MEDIA           ALCISTA        
VolatilTech     +33.35%          MUY ALTA        ALCISTA        
DeclineCorp     -33.81%          MEDIA           BAJISTA        


###  Conclusión Ejecutiva: Perfiles de Inversión
A partir del análisis automatizado del portafolio, podemos emitir las siguientes recomendaciones basadas en datos:

1. **ACME Corp (Perfil Conservador / Recuperación):** A pesar de un rendimiento acumulado negativo (-7.75%), el algoritmo detectó un cambio reciente hacia una tendencia *ALCISTA*. Al tener volatilidad *MEDIA*, es un candidato a mantener en observación para confirmar si el cambio de tendencia es sostenible.
2. **VolatilTech (Perfil Agresivo):** Es la clara ganadora en rendimiento (+33.35%), pero su nivel de volatilidad *MUY ALTA* indica que este rendimiento viene acompañado de un riesgo significativo. Recomendada solo para portafolios con alta tolerancia al riesgo mediante estrategias de *Stop-Loss* ajustadas.
3. **DeclineCorp (Descarte):** Presenta métricas deficientes en todos los rubros (-33.81% de pérdida y tendencia *BAJISTA*). El sistema recomienda liquidar posiciones o evitar compras en este activo hasta que indicadores como el RSI o las Bandas de Bollinger muestren señales claras de reversión.